# 🔴 Notebook 02 — Classical CV Baseline

**CO543/CO5430 — Traffic Sign Detection**

**Pipeline:**
```
Input Image → HSV Conversion → Colour Thresholding (red/blue/yellow)
           → Morphological Cleanup → Contour Detection
           → Bounding Box Proposals → Evaluation
```

**M2 Deliverable**: Precision / Recall numbers on the GTSDB test split.

## 0. Imports & Configuration

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.classical_detector import ClassicalDetector, ClassicalConfig
from src.utils.metrics import compute_iou, compute_precision_recall, evaluate_classical_baseline
from src.utils.visualization import plot_pr_curve

TEST_DIR  = PROJECT_ROOT / 'data' / 'processed' / 'gtsdb' / 'test'
TRAIN_DIR = PROJECT_ROOT / 'data' / 'processed' / 'gtsdb' / 'train'

CLASS_NAMES = {0: 'Prohibitory', 1: 'Danger', 2: 'Mandatory'}
print('Imports OK')

## 1. Understand the HSV Thresholds

In [ ]:
# Visualise the HSV colour masks on a sample image
sample_imgs = sorted((TEST_DIR / 'images').glob('*.jpg'))[:1]
if not sample_imgs:
    print('No test images found — run the GTSDB converter first.')
else:
    img_bgr = cv2.imread(str(sample_imgs[0]))
    hsv     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Default config
    cfg = ClassicalConfig()

    # Red mask (wraps around hue wheel)
    red_m1 = cv2.inRange(hsv, np.array(cfg.red_lower1), np.array(cfg.red_upper1))
    red_m2 = cv2.inRange(hsv, np.array(cfg.red_lower2), np.array(cfg.red_upper2))
    red_mask    = cv2.bitwise_or(red_m1, red_m2)
    blue_mask   = cv2.inRange(hsv, np.array(cfg.blue_lower),   np.array(cfg.blue_upper))
    yellow_mask = cv2.inRange(hsv, np.array(cfg.yellow_lower), np.array(cfg.yellow_upper))

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original')
    axes[1].imshow(red_mask,    cmap='Reds');   axes[1].set_title('Red Mask (Prohibitory)')
    axes[2].imshow(blue_mask,   cmap='Blues');  axes[2].set_title('Blue Mask (Mandatory)')
    axes[3].imshow(yellow_mask, cmap='YlOrBr'); axes[3].set_title('Yellow Mask (Danger)')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'results' / 'figures' / 'classical_hsv_masks.png', dpi=150)
    plt.show()

## 2. Detect on a Single Sample Image

In [ ]:
detector = ClassicalDetector()  # uses default ClassicalConfig

if sample_imgs:
    img_bgr     = cv2.imread(str(sample_imgs[0]))
    detections  = detector.detect(img_bgr)
    annotated   = detector.visualize(img_bgr, detections)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    ax1.imshow(cv2.cvtColor(img_bgr,   cv2.COLOR_BGR2RGB)); ax1.set_title('Original')
    ax2.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); ax2.set_title(f'Detections ({len(detections)} found)')
    ax1.axis('off'); ax2.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'Detections: {len(detections)}')
    for d in detections:
        print(f'  Class: {CLASS_NAMES.get(d[4], d[4])} | Box: ({d[0]},{d[1]}) → ({d[2]},{d[3]})')

## 3. Tune HSV Thresholds Interactively

Adjust the values below, re-run the cell, and observe the mask changes.

In [ ]:
# ── TUNE THESE VALUES ──────────────────────────────────────────────────
custom_cfg = ClassicalConfig(
    red_lower1   = [0,   120, 60],
    red_upper1   = [8,   255, 255],
    red_lower2   = [165, 120, 60],
    red_upper2   = [180, 255, 255],
    blue_lower   = [100, 100, 60],
    blue_upper   = [130, 255, 255],
    yellow_lower = [18,  100, 100],
    yellow_upper = [32,  255, 255],
    min_area     = 500,
    max_area     = 60_000,
)

custom_detector = ClassicalDetector(config=custom_cfg)

if sample_imgs:
    img_bgr    = cv2.imread(str(sample_imgs[0]))
    detections = custom_detector.detect(img_bgr)
    annotated  = custom_detector.visualize(img_bgr, detections)
    fig, ax = plt.subplots(1,1,figsize=(10,5))
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Custom Config — {len(detections)} detections')
    ax.axis('off'); plt.show()

## 4. Evaluate on Full Test Set

In [ ]:
print('Running classical baseline on test set...')
metrics = evaluate_classical_baseline(TEST_DIR)

print('\n── Classical CV Baseline Results (Test Split) ──')
print(f"  Precision : {metrics['precision']:.4f}")
print(f"  Recall    : {metrics['recall']:.4f}")
print(f"  F1 Score  : {metrics['f1']:.4f}")
print(f"  AP@0.5    : {metrics['ap']:.4f}")
print(f"  Detections: {metrics['n_detections']}")
print(f"  GT Boxes  : {metrics['n_ground_truths']}")

# Save results
import json
out_path = PROJECT_ROOT / 'results' / 'metrics' / 'classical_baseline.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump({'model': 'classical', **metrics}, f, indent=2)
print(f'\nResults saved → {out_path}')

## 5. Qualitative: Successes & Failures

In [ ]:
# Load ground truth and detector output for the first 9 test images
img_dir = TEST_DIR / 'images'
lbl_dir = TEST_DIR / 'labels'
img_paths = sorted(img_dir.glob('*.jpg'))[:9]

fig, axes = plt.subplots(3, 3, figsize=(16, 11))
fig.suptitle('Classical Baseline — Test Predictions (Green=Pred, Red=GT)', fontsize=13)

for ax, img_path in zip(axes.flatten(), img_paths):
    img   = cv2.imread(str(img_path))
    if img is None: continue
    h, w  = img.shape[:2]
    rgb   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    preds = detector.detect(img)

    # Draw ground truth (red dashed)
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    xc,yc,bw,bh = map(float, parts[1:])
                    x1=int((xc-bw/2)*w); y1=int((yc-bh/2)*h)
                    rect = patches.Rectangle((x1,y1), int(bw*w), int(bh*h),
                                             linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
                    ax.add_patch(rect)

    # Draw predictions (green solid)
    for (px1,py1,px2,py2,cls) in preds:
        rect = patches.Rectangle((px1,py1), px2-px1, py2-py1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)

    ax.imshow(rgb); ax.set_title(img_path.stem, fontsize=7); ax.axis('off')

plt.tight_layout()
out = PROJECT_ROOT / 'results' / 'qualitative_examples' / 'classical_baseline_predictions.png'
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=150)
plt.show()
print(f'Saved → {out}')

## 6. Analysis & Limitations

Fill in after running the evaluation above:

| Issue | Observation |
|---|---|
| False positives | _(e.g. round red objects like tail-lights)_ |
| False negatives | _(e.g. faded signs, night scenes)_ |
| Speed | _(FPS estimate on test set)_ |

**Why a deep fine-tuned detector is needed:**  
_(write your analysis here after seeing the results)_